## Example 1: An agent runs on a scripted model (no network)

In [ ]:
from strands.models.model import Model


class Scripted(Model):  # replays canned turns, so the agent loop runs without a network
    def __init__(self, turns): self.turns, self.calls = list(turns), []
    def update_config(self, **kw): pass
    def get_config(self): return {}
    async def structured_output(self, *a, **kw): raise NotImplementedError
    async def stream(self, messages, *args, **kwargs):
        self.calls.append([m["role"] for m in messages])
        for event in self.turns.pop(0): yield event

import importlib.metadata

from strands import Agent

# Example 1: the Agent loop, callback-free, over a model that answers one canned turn
assert importlib.metadata.version("strands-agents") == "1.64.0"
agent = Agent(model=Scripted([[{"messageStart": {"role": "assistant"}}, {"contentBlockDelta": {"delta": {"text": "Paris"}}}, {"contentBlockStop": {}}, {"messageStop": {"stopReason": "end_turn"}}]]), callback_handler=None, system_prompt="Answer briefly.")
result = agent("Capital of France?")
assert str(result).strip() == "Paris" and result.stop_reason == "end_turn", (str(result), result.stop_reason)
assert [m["role"] for m in agent.messages] == ["user", "assistant"]
print("Example 1 OK:", str(result).strip())

## Example 2: Tools from plain functions

In [ ]:
from strands import Agent, tool


@tool
def add(a: int, b: int) -> int:
    """Add two integers.

    Args:
        a: first number
        b: second number
    """
    return a + b


# Example 2: @tool builds the JSON schema from the signature and docstring; agent.tool.<name>() calls it directly
spec = add.tool_spec
assert spec["name"] == "add" and spec["description"].startswith("Add two integers")
assert spec["inputSchema"]["json"]["properties"]["a"]["type"] == "integer", spec["inputSchema"]
agent = Agent(tools=[add], callback_handler=None, model="us.amazon.nova-micro-v1:0")
out = agent.tool.add(a=2, b=3)
assert out["status"] == "success" and out["content"][0]["text"] == "5", out
print("Example 2 OK:", out["content"][0]["text"])

## Example 3: The model calls a tool, the agent runs it and continues

In [ ]:
from strands.models.model import Model


class Scripted(Model):  # replays canned turns, so the agent loop runs without a network
    def __init__(self, turns): self.turns, self.calls = list(turns), []
    def update_config(self, **kw): pass
    def get_config(self): return {}
    async def structured_output(self, *a, **kw): raise NotImplementedError
    async def stream(self, messages, *args, **kwargs):
        self.calls.append([m["role"] for m in messages])
        for event in self.turns.pop(0): yield event

from strands import Agent, tool

calls = []
weather = tool(lambda city: calls.append(city) or f"{city}: 21C, sunny", name="weather", description="Weather for a city",
               inputSchema={"json": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}})
# Example 3: turn 1 asks for a tool, turn 2 answers with its result
use = [{"messageStart": {"role": "assistant"}}, {"contentBlockStart": {"start": {"toolUse": {"toolUseId": "t1", "name": "weather"}}}},
       {"contentBlockDelta": {"delta": {"toolUse": {"input": '{"city": "Lisbon"}'}}}}, {"contentBlockStop": {}}, {"messageStop": {"stopReason": "tool_use"}}]
model = Scripted([use, [{"messageStart": {"role": "assistant"}}, {"contentBlockDelta": {"delta": {"text": "Lisbon is sunny, 21C."}}}, {"contentBlockStop": {}}, {"messageStop": {"stopReason": "end_turn"}}]])
result = Agent(model=model, tools=[weather], callback_handler=None)("Weather in Lisbon?")
assert calls == ["Lisbon"] and "sunny" in str(result), (calls, str(result))
assert model.calls[-1] == ["user", "assistant", "user"]  # the tool result went back as a user turn
print("Example 3 OK:", str(result).strip())

## Example 4: Sessions persist a conversation to disk

In [ ]:
from strands.models.model import Model


class Scripted(Model):  # replays canned turns, so the agent loop runs without a network
    def __init__(self, turns): self.turns, self.calls = list(turns), []
    def update_config(self, **kw): pass
    def get_config(self): return {}
    async def structured_output(self, *a, **kw): raise NotImplementedError
    async def stream(self, messages, *args, **kwargs):
        self.calls.append([m["role"] for m in messages])
        for event in self.turns.pop(0): yield event

import tempfile

from strands import Agent
from strands.session import FileSessionManager

# Example 4: a FileSessionManager saves every message; a new agent with the same session id resumes it
folder = tempfile.mkdtemp()
first = Agent(model=Scripted([[{"messageStart": {"role": "assistant"}}, {"contentBlockDelta": {"delta": {"text": "Noted."}}}, {"contentBlockStop": {}}, {"messageStop": {"stopReason": "end_turn"}}]]), callback_handler=None, session_manager=FileSessionManager(session_id="s1", storage_dir=folder))
first("My name is Ada.")
second = Agent(model=Scripted([]), callback_handler=None, session_manager=FileSessionManager(session_id="s1", storage_dir=folder))
assert [m["role"] for m in second.messages] == ["user", "assistant"], second.messages
assert second.messages[0]["content"][0]["text"] == "My name is Ada."
print("Example 4 OK:", len(second.messages), "messages restored")

## Example 5: Hooks observe the agent loop

In [ ]:
from strands.models.model import Model


class Scripted(Model):  # replays canned turns, so the agent loop runs without a network
    def __init__(self, turns): self.turns, self.calls = list(turns), []
    def update_config(self, **kw): pass
    def get_config(self): return {}
    async def structured_output(self, *a, **kw): raise NotImplementedError
    async def stream(self, messages, *args, **kwargs):
        self.calls.append([m["role"] for m in messages])
        for event in self.turns.pop(0): yield event

from strands import Agent
from strands.hooks import AfterInvocationEvent, BeforeInvocationEvent, BeforeModelCallEvent, HookProvider

class Recorder(HookProvider):  # Example 5: a hook provider registers callbacks for lifecycle events
    events = []
    def register_hooks(self, registry, **kwargs):
        for kind in (BeforeInvocationEvent, BeforeModelCallEvent, AfterInvocationEvent):
            registry.add_callback(kind, lambda event: self.events.append(type(event).__name__))
recorder = Recorder()
Agent(model=Scripted([[{"messageStart": {"role": "assistant"}}, {"contentBlockDelta": {"delta": {"text": "ok"}}}, {"contentBlockStop": {}}, {"messageStop": {"stopReason": "end_turn"}}]]), callback_handler=None, hooks=[recorder])("ping")
assert recorder.events == ["BeforeInvocationEvent", "BeforeModelCallEvent", "AfterInvocationEvent"], recorder.events
print("Example 5 OK:", recorder.events)